# Домашнє завдання 9 · Крос-камерний трекер: один ID у двох дронах
**Заняття 9 · MOT, ReID, MTMC · ≈ 3 години · 20 балів**

Збираємо систему зі слайда «Домашнє завдання»: трекер на кожній камері → треклети → зшивка між камерами
за трьома доказами (візуальний опис, час, простір) → глобальні ID і три зони порогів → чесний замір.
Дві камери — два дрони датасету **MDMT**, що одночасно знімають те саме перехрестя; розмітка MDMT дає кожній машині
**той самий ID в обох дронах** — це наш GT для крос-камерної метрики. У дужках — назви слайдів.

| Частина | Що робимо | Слайди |
|---|---|---|
| **1 · трекер на камері** | життєвий цикл треку в секундах → ByteTrack на кожному дроні → HOTA через TrackEval | «Життєвий цикл треку», «ByteTrack: пороги, які реально крутити» |
| **2 · візуальний опис** | кропи з урахуванням піксельного бюджету → ембеддинг треклета (DINOv2 і колір) → наскільки візуальний опис розрізняє машини | «Піксельний бюджет для ReID», «ReID: постановка, бекбони», «Доменний зсув і межі візуального опису» |
| **3 · час і простір** | часове вікно → гомографія між дронами (SIFT + RANSAC) → відстань між треклетами в довжинах машини | «Крос-камерний трекінг: постановка», «Синхронізація часу та геоприв'язка», «Geo-MOT» |
| **4 · глобальні ID** | злиття доказів → угорський алгоритм → три зони порогів, обрані на dev | «Асоціація: вартість і призначення», «Метрики ReID і калібрування порогів» |
| **5 · чесний замір** | ті самі пороги на test: HOTA, ID persistence, хибні зшивки, години оператора, FPS → абляція | «Як міряти трекер на своєму домені», «Бюджет обчислень на edge-вузлі» |
| **6 · три провали** | що, чому, що робити | «Коли трек уже мертвий», «Доменний зсув і межі візуального опису» |

```
дрон 1 ─► детектор ─► ByteTrack ─► треклети ─┐                        ┌─► auto: спільний ID
                                            ├─► час × простір × вигляд ─► угорський ─┼─► оператор
дрон 2 ─► детектор ─► ByteTrack ─► треклети ─┘     (вікно) (гомографія) (ReID)         └─► нова ціль
                                        пороги — на dev (пара 46), замір — на test (пара 48)
```

**Чому детектор симульований.** Готовий COCO-детектор (YOLO11m) на цих кадрах знаходить ≤ 4 % машин: надир, сутінки,
30–70 px — доменний зсув зі слайда 34. Тож детекції генеруються з GT MDMT із реальними оклюзіями (як у §7 демо 1):
перекриті бокси слабкі й обрізані, є пропуски й хибні спрацювання. Свій детектор — у бонусі.

**Як працювати.** Дані, трекер, кропи, SIFT, метрики й малювання вже написано (спільний код — `lessons/09-mot-reid/mtmc.py`).
Вам лишається вписати там, де стоїть `TODO`, **один-два рядки або значення параметра**. Після кожного кроку є `check(...)`:
✅ — крок виконано, ❌ — треба ще покрутити (підказка поруч). Відповіді — у клітинках «✍️», коротко, своїми словами.

**Що здати** — **один архів** `dz9_<прізвище>.zip` в особистому кабінеті LMS: ноутбук **з виводами** і `outputs/report.json`.
Архів збирає остання клітинка ноутбука — розділ «Здача».

**Середовище й дані.** Ядро **Python (cv-advanced)**. Один раз, з кореня репозиторію, з мережею (≈ 1 ГБ):
```bash
python tools/fetch_mdmt.py --seqs 46 48                          # dev і test, ≈ 0,9 ГБ
python tools/fetch_dino_models.py --models dinov2 --size base    # DINOv2-base, ≈ 350 МБ
```
GPU не обов'язковий: повний прогін — ≈ 1 хв на GPU і ≈ 2–5 хв на CPU.

| Крок | Бали |
|---|---|
| 1.1 · життєвий цикл треку в секундах | 1 |
| 2.1 · кропи з урахуванням піксельного бюджету | 1 |
| 2.2 · ембеддинг треклета, вибір дескриптора + висновок | 2 |
| 3.1 · часове вікно | 2 |
| 3.2 · проєкція між дронами | 3 |
| 4.1 · злиття доказів і угорський алгоритм | 2 |
| 4.2 · три зони порогів на dev + обґрунтування | 3 |
| 5.1 · замір на test + висновок | 1 |
| 5.2 · абляція + висновок | 2 |
| 6 · три провали | 3 × 1 |
| **Разом** | **20** |

Кроки 1.1–5.2 зараховано, якщо їхній `check()` дає ✅ і є відповідь у «✍️» там, де вона потрібна. Бонус не оцінюється.

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# спільний код заняття 9: дані MDMT, TrackEval, трекер, кропи, гомографія, метрики
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lessons' / '09-mot-reid').is_dir())
sys.path.insert(0, str(REPO / 'lessons' / '09-mot-reid'))
import mtmc as M
from mdmt import FPS, frame_list, has_trackeval, run_trackeval
from panels import set_style

set_style()
DEV, TEST = '46', '48'            # пороги обираємо на dev (val), міряємо на test — як промпт у ДЗ 6
for seq in (DEV, TEST):
    assert len(frame_list(seq, 1)) and len(frame_list(seq, 2)), f'немає пари {seq}: python tools/fetch_mdmt.py --seqs {seq}'
TE = has_trackeval()              # HOTA через офіційний TrackEval (є в requirements.txt)
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.precision', 3)
print(f'dev {DEV}: {len(frame_list(DEV, 1))} кадрів, test {TEST}: {len(frame_list(TEST, 1))} кадрів, {FPS} FPS | TrackEval: {TE}')


def check(ok, good, bad):
    print(('✅ ' + good) if ok else ('❌ ' + bad))

# Частина 1 · Трекер на кожній камері

## 1.1 · Життєвий цикл треку — у секундах (≈ 10 хв)

`n_init` (скільки підтверджень до народження треку) і `max_age` (скільки трек живе без вимірювань) у коді трекера —
**у кадрах**. Але сенс вони мають у секундах: «бюджет на оклюзію» 1 с при 30 FPS — це 30 кадрів, а при 8 FPS — лише 8.
Пастка зі слайда «ByteTrack: пороги, які реально крутити»: без перерахунку при 8 FPS трек живе 3,75 с замість 1 с.

**TODO 1.1.** Напишіть `frames(sec, fps)`: кількість кадрів для інтервалу `sec` секунд, щонайменше 1.

In [ ]:
N_INIT_S = 0.1     # трек народжується, якщо ціль бачили 0,1 с поспіль
MAX_AGE_S = 1.0    # і живе 1 с без вимірювань (оклюзія, пропуск детектора)


def frames(sec, fps=FPS):
    return ...   # TODO 1.1: секунди → кадри, щонайменше 1


check(frames(1.0, 30) == 30 and frames(1.0, 8) == 8 and frames(0.1, 30) == 3 and frames(0.01, 30) == 1,
      f'n_init = {frames(N_INIT_S)} кадри, max_age = {frames(MAX_AGE_S)} кадрів при {FPS} FPS',
      'frames(1.0, 30) має бути 30, frames(1.0, 8) — 8, frames(0.01, 30) — 1')

## 1.2 · ByteTrack на кожному дроні (≈ 5 хв, код готовий)

Трекер — той самий, що в демо 1 (`BYTE (центр)`): Kalman + IoU + угорський алгоритм, другий етап по слабких боксах.
Його вихід ріжемо на **треклети** — шматки траєкторії з одним ID на одній камері. GT-ID треклета (за більшістю кадрів)
потрібен лише для оцінки: у зшивці він не бере участі. HOTA на кожній камері рахує TrackEval.

In [ ]:
def track_pair(seq):
    """Детектор + трекер на обох дронах пари; треклети й GT для оцінки."""
    out = {'seq': seq, 'ms': {}}
    for cam in (1, 2):
        gt, dets = M.detect_mdmt(seq, cam, seed=0)
        t0 = time.perf_counter()
        hyp = M.run_tracker(dets, n_init=frames(N_INIT_S), max_age=frames(MAX_AGE_S))
        out['ms']['трекер'] = out['ms'].get('трекер', 0) + 1000 * (time.perf_counter() - t0)
        out[cam] = dict(gt=gt, hyp=hyp, tracklets=M.build_tracklets(hyp, cam, gt))
    out['A'], out['B'] = out[1]['tracklets'], out[2]['tracklets']
    out['n_frames'] = len(out[1]['gt']) + len(out[2]['gt'])
    if TE:
        te = run_trackeval({f'{seq}-{c}': out[c]['gt'] for c in (1, 2)},
                           {'bytetrack': {f'{seq}-{c}': out[c]['hyp'] for c in (1, 2)}}, run=f'hw9_{seq}')['bytetrack']
        out['hota_cam'] = {c: te[f'{seq}-{c}'] for c in (1, 2)}
    return out


data = {DEV: track_pair(DEV)}
d = data[DEV]
if TE:
    print(pd.DataFrame(d['hota_cam']).T[['HOTA', 'DetA', 'AssA', 'IDF1', 'IDSW']].rename(index=lambda c: f'{DEV}-{c}'))
print(f"треклетів: дрон 1 — {len(d['A'])}, дрон 2 — {len(d['B'])}; "
      f"спільних GT-цілей: {len({a.gt for a in d['A']} & {b.gt for b in d['B']} - {None})}")
check(not TE or min(r['HOTA'] for r in d['hota_cam'].values()) > 0.6,
      'трекер на кожній камері працює (HOTA > 0.6)', 'HOTA ≤ 0.6 — перевірте frames() у 1.1')

# Частина 2 · Візуальний опис (ReID-базлайн)

## 2.1 · Кропи і піксельний бюджет (≈ 10 хв)

Для візуального опису треклета беремо до `K` кропів, рівномірно по його довжині. Кроп, де ціль менша за `MIN_SIDE` пікселів
по меншій стороні, ембеддинг не допоможе (слайд «Піксельний бюджет для ReID»: тип об'єкта — 60–80 px, ReID — 100+ px;
у MDMT машини 30–70 px — уже на межі). Кропи вирізаються по боксу й повертаються довшою стороною вертикально:
дрони бачать ту саму машину під різними кутами, а DINOv2 не інваріантний до повороту.

**TODO 2.1.** У `crop_rows` відберіть індекси кадрів треклета, де менша сторона бокса `tr.boxes[:, 2:]` ≥ `MIN_SIDE`.

In [ ]:
K, MIN_SIDE = 8, 24


def crop_rows(tr):
    """Індекси рядків tr.boxes для кропів: лише достатньо великі бокси, до K рівномірно по треклету."""
    good = ...   # TODO 2.1: індекси, де min(w, h) ≥ MIN_SIDE
    return good[np.linspace(0, len(good) - 1, min(K, len(good))).astype(int)] if len(good) else good


def read_pair_crops(d):
    t0 = time.perf_counter()
    for cam in (1, 2):
        trs = d[cam]['tracklets']
        M.read_crops(d['seq'], trs, {tr.key: crop_rows(tr) for tr in trs})
    d['ms']['кропи'] = 1000 * (time.perf_counter() - t0)


read_pair_crops(d)
rows = [crop_rows(tr) for tr in d['A'] + d['B']]
small_ok = all((tr.boxes[r, 2:].min(1) >= MIN_SIDE).all() for tr, r in zip(d['A'] + d['B'], rows) if len(r))
with_crops = np.mean([len(tr.crops) > 0 for tr in d['A'] + d['B']])
check(small_ok and with_crops >= 0.8,
      f'кропи є в {with_crops:.0%} треклетів; решта завжди дрібніші за {MIN_SIDE} px — для них візуального опису немає',
      f'у кропи потрапили бокси, менші за {MIN_SIDE} px, або кропів майже немає')
M.show_pairs(d['A'], d['B'], [(i, i) for i in range(6)], title='кропи: перші треклети дронів 1 і 2 (не пари!)')

## 2.2 · Ембеддинг треклета: DINOv2 проти кольору (≈ 15 хв)

Візуальний опис треклета — **середнє по галереї його кропів**, а не один кадр (слайд «Оклюзії: чотири типи»: «порівнюємо
з галереєю всього треку»). Дескриптори два: DINOv2 — baseline ReID без навчання зі слайда «ReID: постановка, бекбони»,
і проста колірна гістограма HSV. Якість міряємо **серед одночасно видимих кандидатів**: ROC AUC «та сама / різні»
і Rank-1 — чи найсхожіший кандидат справді та сама машина.

**TODO 2.2а.** `track_embedding(E)`: середнє рядків `E` (ембеддинги кропів, кожен L2-нормований) і знову L2-нормування.
**TODO 2.2б.** `APP` — назва дескриптора з більшим AUC на dev: його й використаємо далі.

In [ ]:
enc = M.dino_encoder('base')   # DINOv2-base; кропи маленькі (56×112), тож і на CPU це хвилини


def track_embedding(E):
    m = ...                     # TODO 2.2а: середнє по кропах
    return ...                # TODO 2.2а: L2-нормування


def embed_pair(d):
    t0 = time.perf_counter()
    for tr in d['A'] + d['B']:
        if not tr.crops:
            tr.emb = {}
            continue
        tr.emb = {'dino': track_embedding(enc.embed(tr.crops, size=(56, 112))),
                  'hsv': track_embedding(M.hsv_hist(tr.crops))}
    d['ms']['ембеддинги'] = 1000 * (time.perf_counter() - t0)
    for name in ('dino', 'hsv'):
        dim = len(next(tr.emb[name] for tr in d['A'] + d['B'] if tr.emb))
        Ea = np.array([a.emb[name] if a.emb else np.zeros(dim) for a in d['A']])   # без кропів — нульовий вектор
        Eb = np.array([b.emb[name] if b.emb else np.zeros(dim) for b in d['B']])
        d[f'S_{name}'] = np.clip(Ea @ Eb.T, 0, 1)
    d['G'] = M.same_target(d['A'], d['B'])
    d['ov'] = M.time_overlap(d['A'], d['B'])


embed_pair(d)
covis = d['ov'] > 0
q = pd.DataFrame({name: M.app_quality(d[f'S_{name}'], d['G'], covis) for name in ('dino', 'hsv')}).T
print(q)
APP = ...   # TODO 2.2б: 'dino' або 'hsv' — у кого більший AUC на dev
norms = [np.linalg.norm(e) for tr in d['A'] + d['B'] for e in tr.emb.values()]
check(np.allclose(norms, 1, atol=1e-4) and APP == q.AUC.idxmax(),
      f'ембеддинги нормовані; на dev кращий «{APP}»: AUC {q.AUC[APP]:.2f}, Rank-1 {q.Rank1[APP]:.2f} серед ≈ {q.cand_per_track[APP]:.0f} кандидатів',
      'середнє має бути L2-нормоване, а APP — дескриптор із більшим AUC у таблиці')

✍️ **Висновок 2.2** (1–2 речення, входить у бали кроку): чому навіть кращий дескриптор візуального опису лише зрідка ставить
правильну машину на перше місце? Спирайтеся на розмір машин у пікселях, на те, як вони виглядають згори, і на слайди
«Піксельний бюджет для ReID» і «Доменний зсув і межі візуального опису».

✍️ *Ваша відповідь:*

# Частина 3 · Час і простір

## 3.1 · Часове вікно (≈ 15 хв)

Дрони знімають одночасно, тож та сама машина існує в обох потоках у ті самі секунди. `ov[i, j]` — перекриття інтервалів
існування треклетів у секундах: > 0 — бачили одночасно, < 0 — між ними розрив. Кандидатами лишаємо пари з розривом
не більше `GAP_S`: машина могла пройти з поля зору одного дрона в поле іншого (слайд «Крос-камерний трекінг: постановка»:
без просторово-часового фільтра задача «просто не рахується»).

**TODO 3.1.** `time_window(ov, gap_s)` — булева матриця кандидатів.

In [ ]:
GAP_S = 1.0


def time_window(ov, gap_s=GAP_S):
    return ...   # TODO 3.1: кандидат, якщо розрив між треклетами ≤ gap_s секунд


d['W'] = time_window(d['ov'])
G = d['G']
kept, share = G[d['W']].sum() / max(G.sum(), 1), d['W'].mean()
print(f"кандидатів: {d['W'].sum()} з {d['W'].size} пар ({share:.0%}); правильних пар лишилось {G[d['W']].sum()} з {G.sum()} ({kept:.0%})")
probe = time_window(np.array([[-2.0, -1.0, -0.5, 0.0, 3.0]]), 1.0)
check(probe.dtype == bool and probe.tolist() == [[False, True, True, True, True]] and kept >= 0.9,
      f'вікно відсіяло {1 - share:.0%} пар і зберегло {kept:.0%} правильних',
      'розрив 2 с при gap_s = 1 — не кандидат, розрив 1 с і менше або перекриття — кандидат')

## 3.2 · Простір: гомографія між дронами (≈ 25 хв)

Обидва дрони дивляться згори на ту саму дорогу, тож точки землі в їхніх кадрах пов'язує **гомографія** (заняття 2).
Її оцінюємо з самих кадрів, без розмітки: SIFT-точки дрона 1 ↔ дрона 2 + RANSAC (заняття 3). Рахуємо її кожен 10-й кадр;
якщо узгоджених точок (inliers) менше 15, геометрії на цьому відрізку немає — це «гейт якості» доказу.
Тоді центр треклета дрона 1 можна перенести в кадр дрона 2 і порівняти з треклетом дрона 2 **у довжинах машини**
(так відстань не залежить від висоти дрона; слайд «Geo-MOT: трекінг у метрах і його похибка»).

**TODO 3.2.** У `project` перенесіть центр `(cx, cy)` гомографією `Hm`: однорідні координати → множення → ділення на третю.

In [ ]:
def homographies(d):
    t0 = time.perf_counter()
    d['Hs'], ts, inl = M.inter_drone_homographies(d['seq'])
    d['ms']['гомографія'] = 1000 * (time.perf_counter() - t0)
    return ts, inl


ts, inl = homographies(d)
plt.figure(figsize=(14, 3))
plt.plot(ts / FPS, inl, 'o-', ms=3)
plt.axhline(15, ls='--', color='#F87171', lw=1)
plt.xlabel('с'); plt.ylabel('inliers SIFT + RANSAC'); plt.title(f'Пара {DEV}: наскільки надійна гомографія між дронами (нижче пунктиру — немає)')
plt.show()
print(f"геометрія є для {len(d['Hs'])} з {len(frame_list(DEV, 1))} кадрів")

In [ ]:
def project(tr, Hs):
    """{кадр: (x, y)} — центр треклета дрона 1 у кадрі дрона 2 для кадрів, де є гомографія."""
    out = {}
    for t, (cx, cy, w, h) in zip(tr.t.tolist(), tr.boxes):
        if t not in Hs:
            continue
        Hm = Hs[t]
        p = ...   # TODO 3.2: однорідні координати центру, помножені на Hm
        out[t] = ...   # TODO 3.2: назад до (x, y)
    return out


def geometry(d):
    t0 = time.perf_counter()
    d['proj'] = {a.key: project(a, d['Hs']) for a in d['A']}
    d['Dg'] = M.geo_matrix(d['A'], d['B'], d['proj'])
    d['ms']['геометрія'] = d['ms'].get('гомографія', 0) + 1000 * (time.perf_counter() - t0)


geometry(d)
Dg, ok = d['Dg'], np.isfinite(d['Dg'])
t_show = int(np.median([t for p in d['proj'].values() for t in p])) if d['proj'] else 0
M.show_projection(DEV, t_show, d['A'], d['B'], d['proj'])
plt.show()
med_true, med_false = np.median(Dg[G & ok]), np.median(Dg[~G & ok])
print(f'відстань, довжин машини: та сама ціль — медіана {med_true:.2f}, різні — {med_false:.2f}; '
      f'геометрія є для {G[ok].sum()} з {G.sum()} правильних пар')
check(med_true < 0.5 and med_false > 2,
      f'проєкція лягає на ту саму машину (≈ {med_true:.2f} довжини), а сусіди — далеко (≈ {med_false:.1f})',
      'хрестики мають лягати в центри машин дрона 2: перевірте однорідні координати й ділення на p[2]')

# Частина 4 · Глобальні ID

## 4.1 · Злиття доказів і призначення (≈ 15 хв)

Кожен доказ окремо недостатній (слайд «Крос-камерний трекінг: постановка»). Зливаємо їх в одну схожість пари `s ∈ [0, 1]`:

```
s = вікно · s_geo · s_app ** W_APP,   s_geo = exp(−d / SIGMA), якщо геометрія є,  інакше NO_GEO
```

- **вікно** (3.1): поза ним `s = 0` — пару навіть не розглядаємо;
- **s_geo**: відстань `d` із 3.2 у довжинах машини; `SIGMA = 0.5` — на відстані пів машини схожість падає в e разів;
- **NO_GEO = 0.5**: без геометрії пара може дійти щонайбільше до зони оператора, але не до автоматичного злиття;
- **s_app ** W_APP**: косинус обраного в 2.2 дескриптора з вагою `W_APP = 0.5`. У логарифмах це сума доказів
  `log s = log s_geo + 0.5 · log s_app`: візуальний опис — найслабший сигнал (слайд «Кейс 1: ризики, метрика, висновок»), тож його вага менша.
  Ще й тому, що косинус DINOv2 не відкалібрований: на іншій сцені той самий рівень схожості означає інше.

Далі — угорський алгоритм (слайд «Асоціація: вартість і призначення»): кожен треклет дрона 1 отримує не більше одного
партнера в дроні 2 і навпаки. **Але спершу — гейт:** пари зі схожістю нижче `gate` у призначення не пускаємо. Без гейта
угорський алгоритм максимізує *суму* схожостей і охоче міняє одну впевнену пару (0.51) на дві слабкі (0.42 + 0.22 = 0.64) —
так правильні пари губляться. Це те саме «Махаланобіс — лише гейт» зі слайда про DeepSORT і «два гейти» в BoT-SORT.

**TODO 4.1а.** Рядок `s_geo` за формулою вище.
**TODO 4.1б.** У `assign`: обнуліть пари нижче `gate`, потім угорський алгоритм, що максимізує сумарну схожість.

In [ ]:
SIGMA, NO_GEO, W_APP = 0.5, 0.5, 0.5


def fuse(d, use_time=True, use_geo=True, use_app=True):
    """Матриця схожості пар; прапорці — для абляції в 5.2."""
    win = d['W'] if use_time else np.ones_like(d['W'])
    Dg = d['Dg'] if use_geo else np.full_like(d['Dg'], np.nan)
    s_geo = ...   # TODO 4.1а: exp(−d / SIGMA), де геометрія є, інакше NO_GEO
    s_app = d[f'S_{APP}'] ** (W_APP if use_app else 0)
    return win * s_geo * s_app


def assign(S, gate):
    Sg = ...     # TODO 4.1б: пари нижче gate — нуль
    r, c = ...   # TODO 4.1б: угорський алгоритм, що максимізує сумарну схожість Sg
    return [(int(i), int(j)) for i, j in zip(r, c) if Sg[i, j] > 0]


d['S'] = S = fuse(d)
G_good = G & (S >= 0.4)   # правильні пари, які геометрія й візуальний опис упевнено підтверджують
for gate in (0.0, 0.4):
    pairs = assign(S, gate)
    rep = M.link_report(d['A'], d['B'], pairs)
    lost = sum(1 for i, j in zip(*np.where(G_good)) if (i, j) not in set(pairs))
    print(f"гейт {gate}: пар {len(pairs)}, правильних упевнених пар втрачено {lost}, persistence {rep['persistence']:.2f}, хибних {rep['false_rate']:.0%}")
one_to_one = len({i for i, _ in pairs}) == len(pairs) == len({j for _, j in pairs})
check(S.min() >= 0 and S.max() <= 1 and one_to_one and all(S[i, j] >= 0.4 for i, j in pairs) and rep['persistence'] > 0.7,
      'схожість у [0, 1]; з гейтом призначення один-до-одного і лише з упевнених пар',
      'перевірте формулу s_geo, обнулення нижче gate і знак у linear_sum_assignment (максимізуємо схожість)')

## 4.2 · Три зони порогів на dev (≈ 25 хв)

Призначення завжди когось «знаходить» (слайд «ReID: постановка»: без зони відмови ReID завжди знаходить ціль).
Тому кожну пару з 4.1 розкладаємо на три зони (слайд «Метрики ReID і калібрування порогів асоціації»):

- `s ≥ T_HIGH` — **автоматичне злиття**: спільний глобальний ID;
- `T_LOW ≤ s < T_HIGH` — **оператор** перевіряє пару (симулюємо: він підтверджує лише правильні);
- `s < T_LOW` — **нова ідентичність**: окремі ID на кожній камері. `T_LOW` — це й гейт із 4.1.

Ширина середньої зони — це години роботи оператора (слайд: «ширина середньої зони = години роботи оператора на добу»;
`operator_h_per_day` — 10 с на пару, якщо такий потік іде цілодобово; на 10-секундному кліпі це число завищене, дивіться на частку).
Таблиця нижче — перебір порогів на **dev**. **Вимоги замовника:** хибних зшивок в автоматичній зоні ≤ 2 %,
до оператора — не більше 10 % пар, і за цих умов — найбільша підсумкова ID persistence.

**TODO 4.2.** Оберіть `T_LOW` і `T_HIGH` за таблицею та гістограмою.

In [ ]:
clip_s = len(frame_list(DEV, 1)) / FPS
grid = np.round(np.arange(0.2, 0.91, 0.05), 2)
sweep = M.sweep_thresholds(S, assign, d['A'], d['B'], lows=grid, highs=grid, clip_s=clip_s)
fit = sweep[(sweep.false_auto <= 0.02) & (sweep.operator_share <= 0.10)]
print('найкращі варіанти за вимогами (dev):')
print(fit.sort_values(['persistence_final', 'operator_share'], ascending=[False, True]).head(8).to_string(index=False))

pairs0 = assign(S, 0.0)
same = np.array([S[i, j] for i, j in pairs0 if G[i, j]])
diff = np.array([S[i, j] for i, j in pairs0 if not G[i, j]])
plt.figure(figsize=(14, 3.2))
plt.hist([same, diff], bins=np.linspace(0, 1, 41), stacked=True, label=['та сама ціль', 'різні цілі'], color=['#4ADE80', '#F87171'])
plt.xlabel('схожість s пари після призначення (без гейта)'); plt.ylabel('пар'); plt.legend(); plt.title(f'Пара {DEV} (dev)')
plt.show()

T_LOW, T_HIGH = ..., ...   # TODO 4.2: пороги за таблицею й гістограмою
pairs = assign(S, T_LOW)
z = M.zones(S, pairs, T_LOW, T_HIGH)
auto = M.link_report(d['A'], d['B'], z['auto'])
final = M.link_report(d['A'], d['B'], M.resolve_operator(d['A'], d['B'], z))
op_share = len(z['operator']) / max(len(pairs), 1)
best = fit.persistence_final.max() if len(fit) else np.nan
print(f"auto {len(z['auto'])} (хибних {auto['false_rate']:.1%}), оператор {len(z['operator'])} пар ({op_share:.0%}), "
      f"нових {len(z['new'])}; persistence: auto {auto['persistence']:.2f} → з оператором {final['persistence']:.2f}")
check(T_LOW < T_HIGH and auto['false_rate'] <= 0.02 and op_share <= 0.10 and final['persistence'] >= best - 0.03,
      'пороги відповідають вимогам і близькі до найкращих на dev',
      'потрібно: T_LOW < T_HIGH, хибних в auto ≤ 2 %, до оператора ≤ 10 % пар, persistence не гірша за найкращу з таблиці більш ніж на 0.03')

✍️ **Обґрунтування порогів** (2–3 речення, входить у бали кроку): чому саме такі `T_LOW` і `T_HIGH`? Що станеться,
якщо підняти `T_HIGH` на 0.1, а що — якщо опустити `T_LOW` до нуля? Назвіть ціну кожного варіанта: хибні зшивки чи години оператора.

✍️ *Ваша відповідь:*

# Частина 5 · Чесний замір на test

## 5.1 · Ті самі пороги на іншій парі (≈ 10 хв, код готовий)

Усе, що обрано на dev (`MIN_SIDE`, `APP`, `GAP_S`, `SIGMA`, `T_LOW`, `T_HIGH`), тепер **заморожено**. Проганяємо пайплайн
на test-парі `48` і рахуємо звіт зі слайда «Домашнє завдання»: HOTA на кожній камері й **по мережі**
(кадри дрона 2 дописані після кадрів дрона 1, ID глобальні — тож TrackEval бачить і зшивку між камерами), ID persistence,
хибні зшивки, години оператора й час кожного етапу.

In [ ]:
def prepare(seq):
    d = track_pair(seq)
    read_pair_crops(d)
    embed_pair(d)
    d['W'] = time_window(d['ov'])
    homographies(d)
    geometry(d)
    return d


def evaluate(d, use_time=True, use_geo=True, use_app=True, trackeval=True):
    S = fuse(d, use_time, use_geo, use_app)
    pairs = assign(S, T_LOW)
    z = M.zones(S, pairs, T_LOW, T_HIGH)
    final_links = M.resolve_operator(d['A'], d['B'], z)
    auto = M.link_report(d['A'], d['B'], z['auto'])
    final = M.link_report(d['A'], d['B'], final_links)
    clip = len(frame_list(d['seq'], 1)) / FPS
    row = dict(persistence_auto=auto['persistence'], persistence_final=final['persistence'], false_auto=auto['false_rate'],
               operator_pairs=len(z['operator']), operator_share=len(z['operator']) / max(len(pairs), 1),
               operator_h_per_day=M.operator_hours(len(z['operator']), clip))
    if TE and trackeval:
        net = M.mtmc_trackeval(d[1]['gt'], d[2]['gt'], d['A'], d['B'], final_links, run=f"hw9_net_{d['seq']}")
        row.update(net_HOTA=net['HOTA'], net_AssA=net['AssA'], net_IDF1=net['IDF1'])
        row.update({f'HOTA_cam{c}': d['hota_cam'][c]['HOTA'] for c in (1, 2)})
    return row, S, pairs, z


data[TEST] = prepare(TEST)
report = {}
for seq, role in ((DEV, 'dev'), (TEST, 'test')):
    report[f'{role} ({seq})'] = evaluate(data[seq])[0]
rep_df = pd.DataFrame(report).T
print(rep_df.to_string())

dt = data[TEST]
n_fr = dt['n_frames']
ms = pd.Series(dt['ms']) / n_fr
print('\nмс на кадр (обидва дрони):', ', '.join(f'{k} {v:.1f}' for k, v in ms.items()),
      f'| разом {ms.sum():.1f} мс → ≈ {1000 / ms.sum():.0f} FPS на {"GPU" if "cuda" in str(enc.device) else "CPU"}')
json.dump(dict(params=dict(N_INIT_S=N_INIT_S, MAX_AGE_S=MAX_AGE_S, MIN_SIDE=MIN_SIDE, APP=APP, GAP_S=GAP_S, SIGMA=SIGMA,
                           NO_GEO=NO_GEO, W_APP=W_APP, T_LOW=T_LOW, T_HIGH=T_HIGH),
               report=rep_df.round(4).to_dict(orient='index'), ms_per_frame=ms.round(2).to_dict()),
          open(OUT_DIR / 'report.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
check((OUT_DIR / 'report.json').exists(), f'звіт збережено в {OUT_DIR / "report.json"}', 'звіт не збережено')

✍️ **Висновок 5.1** (2–3 речення, входить у бали кроку): чи перенеслися пороги з dev на test — що з хибними зшивками
й годинами оператора? Який етап найдорожчий за часом і чи вклалася б система в 15 FPS на Orin
(слайд «Бюджет обчислень на edge-вузлі»)?

✍️ *Ваша відповідь:*

## 5.2 · Абляція: який доказ що дає (≈ 15 хв)

Слайд «Домашнє завдання» вимагає абляцію без часового вікна. Зробимо повнішу: вимикаємо докази по одному й попарно,
пороги ті самі. Порівнюємо на **test**.

**TODO 5.2.** Заповніть прапорці `use_time`, `use_geo`, `use_app` для кожного варіанта.

In [ ]:
VARIANTS = {
    'усі три докази':        dict(use_time=True, use_geo=True, use_app=True),
    'без часового вікна':    dict(...),   # TODO 5.2
    'без геометрії':         dict(...),   # TODO 5.2
    'без вікна й геометрії': dict(...),   # TODO 5.2
    'без візуального опису':       dict(...),   # TODO 5.2
}
abl = pd.DataFrame({k: evaluate(dt, **v, trackeval=False)[0] for k, v in VARIANTS.items()}).T
print(abl[['persistence_auto', 'persistence_final', 'false_auto', 'operator_pairs', 'operator_share']].to_string())
flags = [tuple(v.values()) for v in VARIANTS.values()]
check(len(set(flags)) == 5 and VARIANTS['без часового вікна']['use_time'] is False and VARIANTS['без геометрії']['use_geo'] is False
      and VARIANTS['без вікна й геометрії'] == dict(use_time=False, use_geo=False, use_app=True) and VARIANTS['без візуального опису']['use_app'] is False,
      'абляція виконана', 'кожен варіант вимикає саме той доказ, що в його назві')

✍️ **Висновок 5.2** (2–3 речення, входить у бали кроку): яку роль грає кожен доказ — хто **знаходить** пари,
хто **відсікає** хибні, а хто **зменшує роботу оператора**? Чому «без візуального опису» з'являються хибні зшивки, хоча
persistence майже не падає? Порівняйте з висновками слайдів «Кейс 1: ризики, метрика, висновок» і «Кейс 3».

✍️ *Ваша відповідь:*

# 6 · Три провали (≈ 20 хв)

Нижче — галереї з **test**: хибні автоматичні зшивки, пропущені цілі (обидва дрони бачили машину, а спільного ID немає)
і пари в зоні оператора. Знайдіть **три різні** провали. Для кожного — по одному реченню:
- **що** видно: номери треклетів, час, схожість `s`;
- **чому** — ваша гіпотеза (оклюзія? дрібна машина? немає геометрії на цьому відрізку? однакові машини поруч? трекер розірвав трек?);
- **що робити** — прийом зі слайдів заняття (без донавчання моделі).

In [ ]:
row, S_t, pairs_t, z_t = evaluate(dt, trackeval=False)
A, B, Gt = dt['A'], dt['B'], dt['G']
final_t = M.resolve_operator(A, B, z_t)
wrong_auto = [(i, j) for i, j in z_t['auto'] if not Gt[i, j]]
linked = {A[i].gt for i, j in final_t if Gt[i, j]}
missed = []
for i, a in enumerate(A):
    if a.gt is None or a.gt in linked:
        continue
    js = np.where(Gt[i])[0]
    if len(js):
        missed.append((i, int(js[0])))
M.show_pairs(A, B, wrong_auto, S_t, title=f'хибні автоматичні зшивки: {len(wrong_auto)}')
M.show_pairs(A, B, missed, S_t, title=f'пропущені цілі: {len(missed)} (та сама машина, s її пари)')
M.show_pairs(A, B, z_t['operator'], S_t, title=f"зона оператора: {len(z_t['operator'])} пар")
print('пропущені — чому впала схожість:')
for i, j in missed[:8]:
    print(f"  д1 id{A[i].tid} ↔ д2 id{B[j].tid}: s = {S_t[i, j]:.2f}, геометрія d = {dt['Dg'][i, j]:.2f} довжини, "
          f"у вікні: {bool(dt['W'][i, j])}, s_{APP} = {dt[f'S_{APP}'][i, j]:.2f}")

✍️ *6.1:*

✍️ *6.2:*

✍️ *6.3:*

## Бонус (за бажанням, не оцінюється)
- **Пара без геометрії.** `python tools/fetch_mdmt.py --seqs 22` і прогоніть `prepare('22')` + `evaluate(...)`: дрони
  дивляться під надто різними кутами, SIFT не знаходить гомографії. Що лишається від persistence і куди йдуть пари — в auto
  чи до оператора? Це той самий гейт якості доказу, що на слайді «Коли трек уже мертвий»: деградація по рівнях.
- **Розсинхрон.** Зсуньте кадри дрона 2 на кілька кадрів (`dt[2]['tracklets']` → `t + k`) і повторіть 5.1 — як у демо 02.
  При якому зсуві геометрія перестає рятувати і чому межа залежить від швидкості машин?
- **Свій детектор.** Замініть `M.detect_mdmt` власним (Grounding DINO з ДЗ 6 із промптом `vehicle .`): масив
  `(cx, cy, w, h, score)` на кадр. Як змінилися HOTA на камері і persistence по мережі?
- **CVAT.** Розмітка MDMT — це CVAT XML (`artifacts/mdmt/new_xml/`): імпортуйте її в CVAT разом із кадрами й подивіться
  на свої провали з частини 6 у контексті всього ролика.

# Здача
Усе здається **одним архівом** через особистий кабінет LMS. Порядок:
1. **Kernel → Restart Kernel and Run All Cells**: ноутбук має виконатись від початку до кінця, з усіма ✅.
2. Заповніть відповіді в клітинках «✍️».
3. Збережіть ноутбук (**Ctrl+S**) і запустіть клітинку нижче: вона збере в архів `dz9_<прізвище>.zip` ноутбук і `outputs/report.json`.
4. Прикріпіть **цей один архів** у особистому кабінеті LMS. Окремі файли, посилання на диск чи репозиторій не приймаються.

In [ ]:
import zipfile

SURNAME = ''   # ✍️ прізвище латиницею, наприклад 'shevchenko'
NOTEBOOK = Path('04_mtmc_tracker_student.ipynb')
FILES = [NOTEBOOK, OUT_DIR / 'report.json']

assert SURNAME, 'впишіть прізвище в SURNAME'
missing = [str(f) for f in FILES if not f.exists()]
assert not missing, f'немає файлів {missing}: виконайте всі клітинки й збережіть ноутбук'
archive = Path(f'dz9_{SURNAME}.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        z.write(f, f.as_posix())
    names = z.namelist()
print(f'→ {archive.resolve()} ({archive.stat().st_size / 2**20:.1f} МБ): {", ".join(names)}')
print('Прикріпіть цей архів в особистому кабінеті LMS.')